# Launch Demand Forecaster

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Forecast launch demand** using `SNOWFLAKE.ML.FORECAST`
2. **Model S-curve adoption** patterns
3. **Learn from analogous products** (similar launches)
4. **Predict 24-month penetration** with ML
5. **Plan supply** based on demand forecasts

---

## What You'll Build

A **product launch forecasting system** that predicts demand:
- New product demand forecasting
- S-curve adoption modeling (slow start → rapid growth → plateau)
- Analogous product pattern learning
- 24-month market penetration predictions
- Supply planning recommendations

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `SNOWFLAKE.ML.FORECAST` - Time-series launch predictions
- S-curve modeling - Built into synthetic data generation
- Confidence intervals - Optimistic/pessimistic scenarios
- `DATEADD()` - Create future time periods
- Launch readiness dashboard - Supply vs. demand

Let's begin!

---

## Paso 1: Configuración del Entorno

### Launch Forecasting Challenge

Limited historical data for new products → Use analogous product patterns

In [ ]:
CREATE DATABASE IF NOT EXISTS LAUNCH_FORECAST_DB;
USE SCHEMA LAUNCH_FORECAST_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db;

---

## Paso 2: Create Tables

In [ ]:
CREATE OR REPLACE TABLE historical_launches (
    launch_id VARCHAR(50) PRIMARY KEY,
    product_name VARCHAR(100),
    launch_date DATE,
    therapeutic_area VARCHAR(100),
    months_post_launch INTEGER,
    monthly_scripts INTEGER,
    cumulative_scripts INTEGER
);

CREATE OR REPLACE TABLE new_product_launches (
    product_id VARCHAR(50) PRIMARY KEY,
    product_name VARCHAR(100),
    expected_launch_date DATE,
    therapeutic_area VARCHAR(100),
    target_peak_share_pct FLOAT
);

---

## Paso 3: Generar Launch History Data

S-curve adoption: slow start → rapid growth → plateau

In [ ]:
-- Historical launches (analogous products)
INSERT INTO historical_launches
WITH products AS (
    SELECT * FROM (VALUES
        ('Xarelto', '2011-07-01', 'Cardiovascular'),
        ('Eylea', '2011-11-01', 'Ophthalmology'),
        ('Stivarga', '2012-09-01', 'Oncology')
    ) t(product, launch_date, therapeutic_area)
),
months_post AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 as month_num
    FROM TABLE(GENERATOR(ROWCOUNT => 36))
)
SELECT 
    UUID_STRING() as launch_id,
    p.product as product_name,
    p.launch_date::DATE as launch_date,
    p.therapeutic_area,
    m.month_num as months_post_launch,
    -- S-curve adoption pattern
    FLOOR(100000 * (1 / (1 + EXP(-0.15 * (m.month_num - 18)))) * (1 + UNIFORM(-0.1,0.1,RANDOM()))) as monthly_scripts,
    SUM(monthly_scripts) OVER (PARTITION BY p.product ORDER BY m.month_num) as cumulative_scripts
FROM products p
CROSS JOIN months_post m;

-- New launches to forecast
INSERT INTO new_product_launches VALUES
    ('NP001', 'Novel Anticoagulant Factor XIa', '2025-01-01', 'Cardiovascular', 20.0),
    ('NP002', 'Next-Gen Retinal Gene Therapy', '2025-06-01', 'Ophthalmology', 15.0);

SELECT product_name, COUNT(*) as months_data, MAX(cumulative_scripts) as peak_cumulative
FROM historical_launches GROUP BY product_name;

---

## Paso 4: Train Launch Forecast Models

Use historical patterns to predict new launch trajectories

In [ ]:
-- Create training data view for analogous product (Xarelto)
CREATE OR REPLACE VIEW launch_training_data AS
SELECT 
    DATEADD(month, months_post_launch, launch_date) as forecast_date,
    monthly_scripts
FROM historical_launches 
WHERE product_name='Xarelto';

-- Train forecast using analogous product
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST launch_demand_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'launch_training_data'),
    TIMESTAMP_COLNAME => 'forecast_date',
    TARGET_COLNAME => 'monthly_scripts'
);

SELECT 'Launch forecast model trained!' as status;

---

## Paso 5: Generar 24-Month Launch Forecasts

Predict demand for new products using analogous patterns

In [ ]:
-- Generar 24-month forecast
CREATE OR REPLACE TABLE new_product_forecasts AS
SELECT 
    ts as forecast_month,
    forecast as predicted_scripts,
    lower_bound as conservative_scripts,
    upper_bound as optimistic_scripts
FROM TABLE(launch_demand_forecast!FORECAST(FORECASTING_PERIODS => 24));

-- View forecast
SELECT 
    forecast_month,
    ROUND(predicted_scripts, 0) as predicted,
    ROUND(conservative_scripts, 0) as conservative,
    ROUND(optimistic_scripts, 0) as optimistic
FROM new_product_forecasts
WHERE MONTH(forecast_month) IN (1,6,12,18,24)
ORDER BY forecast_month
LIMIT 10;

---

## Paso 6: Launch Readiness Analysis

In [ ]:
-- Calculate launch metrics
CREATE OR REPLACE VIEW launch_readiness_metrics AS
WITH forecast_summary AS (
    SELECT 
        SUM(predicted_scripts) as total_year1_scripts,
        AVG(predicted_scripts) as avg_monthly_scripts,
        MAX(predicted_scripts) as peak_monthly_scripts
    FROM new_product_forecasts
    WHERE forecast_month < DATEADD('year', 1, CURRENT_DATE())
)
SELECT 
    'Novel Anticoagulant Factor XIa' as product,
    total_year1_scripts,
    ROUND(total_year1_scripts / 1000000, 2) as scripts_millions,
    ROUND(peak_monthly_scripts, 0) as peak_monthly,
    -- Supply planning
    CEIL(total_year1_scripts * 1.15) as manufacturing_target,
    CASE 
        WHEN total_year1_scripts >= 2000000 THEN '🟢 Strong Launch'
        WHEN total_year1_scripts >= 1000000 THEN '🟡 Moderate Launch'
        ELSE '🔴 Weak Launch'
    END as launch_strength
FROM forecast_summary;

SELECT * FROM launch_readiness_metrics;

---

## Paso 7: Interactive Launch Dashboard

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🚀 Launch Demand Forecaster")

# Launch readiness
metrics = session.sql("SELECT * FROM launch_readiness_metrics").collect()[0]

col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Year 1 Scripts", f"{metrics['SCRIPTS_MILLIONS']:.2f}M")
with col2:
    st.metric("Peak Monthly", f"{int(metrics['PEAK_MONTHLY']):,}")
with col3:
    st.metric("Launch Strength", metrics['LAUNCH_STRENGTH'])

# Forecast chart
st.subheader("📈 24-Month Demand Forecast")

forecast = session.sql("""
    SELECT forecast_month, 
           predicted_scripts as predicted,
           conservative_scripts as conservative,
           optimistic_scripts as optimistic
    FROM new_product_forecasts ORDER BY forecast_month
""").to_pandas()

st.line_chart(forecast.set_index('FORECAST_MONTH'))
st.caption("Demand in monthly prescriptions - showing scenario range")

# Cumulative adoption
st.subheader("📊 Cumulative Adoption Curve")

cumulative = session.sql("""
    SELECT forecast_month,
           SUM(predicted_scripts) OVER (ORDER BY forecast_month) as cumulative
    FROM new_product_forecasts
""").to_pandas()

st.line_chart(cumulative.set_index('FORECAST_MONTH')['CUMULATIVE'])

# Supply planning
st.subheader("📦 Manufacturing Target")
st.metric("Production Goal", f"{int(metrics['MANUFACTURING_TARGET']):,} units", 
          help="Year 1 forecast + 15% buffer")

---

## ✅ Complete!

### What You Learned

1. ✅ Forecast new product launches with `SNOWFLAKE.ML.FORECAST`
2. ✅ Use analogous product data for training
3. ✅ Model S-curve adoption patterns
4. ✅ Generar scenario planning (optimistic/pessimistic)
5. ✅ Build launch readiness dashboards

### Launch Forecasting Pattern

**S-Curve Adoption:**
- **Months 0-6**: Slow start (awareness building)
- **Months 6-18**: Rapid growth (tipping point)
- **Months 18-36**: Plateau (market saturation)

### Key Applications

- **Supply planning**: Manufacturing targets
- **Sales force sizing**: Territory allocation
- **Budget planning**: Marketing spend
- **Investor guidance**: Revenue projections

[Snowflake ML FORECAST Docs](https://docs.snowflake.com/en/user-guide/ml-functions/forecasting)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `LAUNCH_FORECAST_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS LAUNCH_FORECAST_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;